# Silver Template — Worked Example (Free Edition serverless)

**Copy this pattern for your own Silver tables.**

Uses the smallest source table (`cl_loyalty_tiers`, 4 rows) to demonstrate the canonical Silver build pattern adapted for **Databricks Free Edition serverless**:

1. `%pip install snowflake-connector-python` (no Maven JAR available)
2. Widget-based credentials (no secret scope on Free tier)
3. Read Bronze via shared `read_from_snowflake()` helper
4. PySpark transforms (T1 dates, T3 surrogate keys)
5. `add_anomaly_columns()` + `flag()` for the 4 mandatory columns
6. Write Silver via shared `write_to_snowflake()` helper (uses `snowflake-connector-python.write_pandas`)

**Reference:** `docs/anomaly_taxonomy.md` for reason codes; `docs/glossary.md` for KPI definitions; `docs/databricks_setup.md` for the why-Free-Edition explanation.

## Cell 1 — Install the Snowflake connector

In [ ]:
%pip install -q snowflake-connector-python pandas rapidfuzz
# dbutils.library.restartPython()  # SKIP on Free Edition — kernel hangs

## Cell 2 — Widgets (paste your Snowflake password into `sf_password`)

In [ ]:
dbutils.widgets.text('sf_account',   'rhxendw-yb24678')
dbutils.widgets.text('sf_user',      'NEXAMART_LEAD')
dbutils.widgets.text('sf_password',  '')
dbutils.widgets.text('sf_warehouse', 'NEXAMART_WH')
dbutils.widgets.text('sf_role',      'ACCOUNTADMIN')

## Cell 3 — Imports (shared helpers from `_shared/`)

If the import fails, your repo path is different — adjust `sys.path.append(...)` to point at `notebooks/_shared/` in your workspace.

In [ ]:
import sys
sys.path.append('/Workspace/Repos/<your-org>/nexamart-m1/notebooks/_shared')

from pyspark.sql import functions as F
from utils_dates    import parse_date, is_parse_failure
from utils_keys     import surrogate_key
from utils_anomaly  import add_anomaly_columns, flag, flag_date_parse_failure
from utils_snowflake import read_from_snowflake, write_to_snowflake

## Cell 4 — Read from Bronze

Use `read_from_snowflake(spark, table)` — internally uses `snowflake-connector-python.fetch_pandas_all` + `spark.createDataFrame`. For tables larger than driver memory, push down filters via `where=`.

In [ ]:
bronze = read_from_snowflake(spark, 'cl_loyalty_tiers', schema='NEXAMART_BRONZE')
print(f'Read {bronze.count()} rows from Bronze cl_loyalty_tiers')
bronze.show()

## Cell 5 — T1 (date parsing)

`cl_loyalty_tiers` has no date columns; pattern shown for tables that do:

```python
df = df.withColumnRenamed('order_date', 'order_date_raw')
df = df.withColumn('order_date', parse_date(F.col('order_date_raw'), 'iso_date'))
df = flag_date_parse_failure(df, parsed_col='order_date', raw_col='order_date_raw')
```

Format hints: `ddmmyyyy`, `iso_date`, `iso_timestamp`, `ddmonyyyy`, `slash_datetime`, `unix_epoch`, `pg_timestamp`. See `docs/date_formats.md`.

## Cell 6 — T3 (surrogate key)

Build deterministic SHA-256 SK from natural keys. For `cl_loyalty_tiers`, natural key is `tier_id`.

In [ ]:
silver = bronze.withColumn('loyalty_tier_key', surrogate_key(F.col('tier_id')))
silver.select('tier_id', 'loyalty_tier_key', 'tier_name').show(truncate=False)

## Cell 7 — Add the 4 mandatory anomaly columns

In [ ]:
silver = add_anomaly_columns(silver)
silver.select('tier_id', 'anomaly_flag', 'anomaly_reason_code',
              'data_quality_status', 'metric_certainty_level').show()

## Cell 8 — Apply `flag()` for any anomalies

Demo with a no-op condition (no rows match) so the call pattern is visible. Real Silver notebooks have multiple `flag()` calls — one per detection rule.

```python
silver = flag(silver,
    condition=(F.col('points_per_currency') < 0),
    reason_code='NEGATIVE_AMOUNT',
    status='FLAGGED_ANOMALY',
    certainty='UNRELIABLE')
```

In [ ]:
silver = flag(
    silver,
    condition=(F.col('tier_name') == 'NONEXISTENT'),
    reason_code='NEGATIVE_AMOUNT',
    status='FLAGGED_ANOMALY',
    certainty='UNRELIABLE',
)
silver.select('tier_id', 'anomaly_flag', 'anomaly_reason_code',
              'data_quality_status', 'metric_certainty_level').show()

## Cell 9 — Write to Silver (overwrite for idempotence)

`write_to_snowflake(df, table, schema)` collects to driver via `toPandas()` then uses `write_pandas(...)`. For our scale this is fast enough; first run takes ~30s on Free Edition.

In [ ]:
n = write_to_snowflake(silver, 'silver_loyalty_tiers', schema='NEXAMART_SILVER')
print(f'Wrote silver_loyalty_tiers: {n} rows')

# Round-trip read-back
verify = read_from_snowflake(spark, 'silver_loyalty_tiers', schema='NEXAMART_SILVER')
print(f'Read back {verify.count()} rows.')

## Acceptance contract

Every Silver table you build must:
1. Have all 4 mandatory columns populated for every row.
2. Use only registered `anomaly_reason_code` values from `docs/anomaly_taxonomy.md`.
3. Use `surrogate_key()` for any FK that becomes a Gold dim/fact join.
4. Re-run idempotently (use `write_to_snowflake(..., overwrite=True)` — that's the default).
5. Never delete rows — flag them.

PRs that violate any of the above get rejected.